# Spatio-temporal zonal analysis

In [ ]:
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

import pylandstats as pls

In order to perform zonal analyses over time, pylandstats features an additional `SpatioTemporalZonalAnalysis` analysis class - as well as `SpatioTemporalBufferAnalysis` and `SpatioTemporalZonalGridAnalysis`.

Like in the [spatio-temporal analysis example](spatiotemporal-analysis.ipynb), we will use the three extracts of [Veveyse district](https://en.wikipedia.org/wiki/Veveyse_District) from the [Swiss Land Statistics (SLS) datasets from the Swiss Federal Statistical Office](https://www.bfs.admin.ch/bfs/en/home/services/geostat/swiss-federal-statistics-geodata/land-use-cover-suitability/swiss-land-use-statistics.html) for the years 1980, 1992, 2004 and 2013, yet in this case we also need to specify how the buffers are constructed.

The data used in this notebook ships with the docs in the `data` directory, namely:
- the land use/land cover (LULC) data is downloaded and preprocessed (see [A03-swisslandstats-preprocessing.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A03-swisslandstats-preprocessing.ipynb) for more details).
- the elevation zones vector data is downloaded and preprocessed (see [A04-elevation-zones.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A04-elevation-zones.ipynb) for more details).

## Spatio-temporal zonal analysis

Besides the base LULC maps, we will use a geopackage file defining a set of elevation zones. Like with the `ZonalAnalysis` class, we can use the `zone_index` argument to indicate which column of the geopackage file will be used to index the zones:

In [ ]:
URBAN_CLASS_VAL = 1
input_filepaths = [
    "data/veveyse/LU85_4.tif",
    "data/veveyse/LU97_4.tif",
    "data/veveyse/LU09_4.tif",
    "data/veveyse/LU18_4.tif",
]
years = ["1980", "1992", "2004", "2013"]
elev_zones_filepath = "data/elev-zones.gpkg"

stza = pls.SpatioTemporalZonalAnalysis(
    input_filepaths, elev_zones_filepath, zone_index="elev-zone", dates=years
)

Like `SpatioTemporalAnalysis`, `BufferAnalysis` and/or `ZonalAnalysis`, we can compute the data frame of class metrics through the `compute_class_metrics_df` method:

In [ ]:
class_metrics_df = stza.compute_class_metrics_df()
class_metrics_df.head()

Note that in this case, the data frame features a three-level [MultiIndex](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.MultiIndex.html) that distinguishes the computed value for each class, zone and date. Again, we can operate upon such data frames as we would do with any other pandas data frame. For instance, we might want to evaluate the difference between the proportion of landscape occupied of urban patches (represented by a `class_val` of 1) computed for the elevation zone of "<1000" and ">1500":

In [ ]:
(
    class_metrics_df.loc[(1, "<1000"), "proportion_of_landscape"]
    - class_metrics_df.loc[(1, ">1500"), "proportion_of_landscape"]
)

Likewise in the other classes of pylandstats, if we want to compute the metrics data frame only for a subset of metrics or classes, or customize how the metrics are computed, we must respectively pass the arguments `metrics`, `classes` or `metrics_kwargs` to the `compute_class_metrics_df` and `compute_landscape_metrics_df` methods of `SpatioTemporalZonalAnalysis` as in:

In [ ]:
metrics = ["proportion_of_landscape", "edge_density", "fractal_dimension_am"]
classes = [URBAN_CLASS_VAL]
metrics_kwargs = {
    "proportion_of_landscape": {"percent": False},
    "edge_density": {"count_boundary": True},
}
stza.compute_class_metrics_df(
    metrics=metrics, classes=classes, metrics_kwargs=metrics_kwargs
)

Another important functionality of the `SpatioTemporalZonalAnalysis` is plotting the time series of metrics at each zone. We can accomplish that through the `plot_metric` method. For instance, let us plot the proportion of landscape at the level of the *urban* class (`class_val` of 1):

In [ ]:
stza.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

It can also be interesting to visualize such information in space by using `compute_zonal_statistics_gdf` method to obtain a geo-data frame:

In [ ]:
zonal_statistics_gdf = stza.compute_zonal_statistics_gdf(
    metrics=metrics, class_val=URBAN_CLASS_VAL
)
zonal_statistics_gdf

Note that the columns of the geo-data frame are indexed in two levels, namely the metrics and dates. Since now we have a time series of values, we can spatially plot the evolution of the metrics at each zone:

In [ ]:
num_years = len(years)
figwidth, figheight = plt.rcParams["figure.figsize"]
fig, axes = plt.subplots(1, num_years, figsize=(num_years * figwidth, figheight))

# get min and max values for all years to have the same scale across plots
vmin = zonal_statistics_gdf["proportion_of_landscape"].min().min()
vmax = zonal_statistics_gdf["proportion_of_landscape"].max().max()
for year, ax in zip(years, axes):
    zonal_statistics_gdf.plot(
        ("proportion_of_landscape", year),
        ax=ax,
        alpha=0.8,
        legend=True,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(year)
    cx.add_basemap(
        ax, crs=zonal_statistics_gdf.crs, source=cx.providers.CartoDB.Positron
    )

In fact, we can apply any transformation to the geo-data frame to visualize the metrics as required. For instance, we can transform the data frame to have separate columns for each year (how this is accomplished is beyond the scope of this tutorial), so that we can explore all the metrics values in the map:

In [ ]:
plot_gdf = zonal_statistics_gdf.copy()  # drop("geometry", axis=1).unstack()
plot_gdf.columns = plot_gdf.columns.to_flat_index().map(lambda tup: "-".join(tup))
plot_gdf = gpd.GeoDataFrame(
    plot_gdf,
    geometry=plot_gdf.reset_index()["elev-zone"].map(stza.zone_gser).values,
    crs=stza.zone_gser.crs,
)
plot_gdf.explore()

## Spatiotemporal buffer analysis

Let us now consider three buffers of 2, 4 and 6km around the center of the town of Chatel-St-Denis. We can explore how landscape metrics at such extents change through times by using the `SpatioTemporalBufferAnalysis` class as in:

In [ ]:
# latitude and longitude of the center of Chatel-St-Denis according to OpenStreetMap
base_geom = Point(6.8992073, 46.52634)
base_geom_crs = "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs"

# buffer distances (in meters)
buffer_dists = [2000, 4000, 6000]

stba = pls.SpatioTemporalBufferAnalysis(
    input_filepaths, base_geom, buffer_dists, base_geom_crs=base_geom_crs, dates=years
)

Analogously to `BufferAnalysis`, we could also initialize the from a polygon geometry (such an administrative boundary) by passing such object as the `base_geom` argument. See [the zonal analysis notebook](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/03-zonal-analysis.ipynb) to see how this can be done

The functionalities of `SpatioTemporalBufferAnalysis` are essentially the same as those of `SpatioTemporalZonalAnalysis` reviewed above:

In [ ]:
class_metrics_df = stba.compute_class_metrics_df()
class_metrics_df.head()

In [ ]:
stba.compute_landscape_metrics_df()

Similarly, we can produce the same plot at the landscape level by omitting the `class_val` argument:

In [ ]:
stba.plot_metric("fractal_dimension_am")

In this case, the legend shows the buffer distance that corresponds to the plotted line.

## Spatiotemporal zonal grid analysis

Finally, we can explore the temporal evolution of landscape metrics over a single regular rectangular grid using the `SpatioTemporalZonalGridAnalysis` class:

In [ ]:
zone_width, zone_height = 2000, 2000  # in this case, in meters

stzga = pls.SpatioTemporalZonalGridAnalysis(
    input_filepaths, zone_width=zone_width, zone_height=zone_height, dates=years
)

Likewise with `ZonalAnalysis` we can also define the number of zones that we desire in each dimension by means of the `num_zone_rows` and `num_zone_cols` keyword arguments of the initialization method.

Again, the functionalities of `SpatioTemporalZonalGridAnalysis` are essentially the same as those of `SpatioTemporalZonalAnalysis` (and `SpatioTemporalBufferAnalysis`) reviewed above:

In [ ]:
class_metrics_df = stzga.compute_class_metrics_df(metrics=metrics)
class_metrics_df.head()